In [45]:
import fitz
import re 
import json
import nltk
nltk.download('punkt')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd 
from sqlalchemy import create_engine
import openpyxl

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dipan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [46]:
import fitz
import re
import json
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk

# -----------------------------
# LOAD PDF
# -----------------------------
doc = fitz.open(r"D:\exercises\Thesis\NIST.SP.800-82r3.pdf")

full_text = ""
for page in doc:
    full_text += "\n" + page.get_text()

# -----------------------------
# 🔥 REMOVE FRONT MATTER
# -----------------------------
full_text = re.split(r"\n1\s+[A-Z]", full_text, maxsplit=1)[-1]

# -----------------------------
# 🔥 REMOVE TABLE OF CONTENTS
# -----------------------------
full_text = re.sub(
    r"Table of Contents.*?(?=\n\d+\s+[A-Z])",
    "",
    full_text,
    flags=re.IGNORECASE | re.DOTALL
)

# -----------------------------
# HEADING DETECTION
# -----------------------------
heading_pattern = re.compile(
    r"\n(\d+(?:\.\d+)*\.?)\s+([A-Z][A-Za-z0-9\(\)\-.,: ]{5,})"
)

matches = list(heading_pattern.finditer(full_text))

# -----------------------------
# CLEAN TEXT (🔥 IMPROVED)
# -----------------------------
def clean_content(text: str) -> str:
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # remove citations
    text = re.sub(r"\[[^\]]+\]", " ", text)

    # remove figures
    text = re.sub(r"\bfig(?:ure)?\.?\s*\d+\.?", " ", text, flags=re.IGNORECASE)

    #  remove tables
    text = re.sub(r"\btable\s*\d+\.?", " ", text, flags=re.IGNORECASE)

    #  remove "see ..." references
    text = re.sub(r"\bsee\s+(table|figure|section|https?)\S*", " ", text, flags=re.IGNORECASE)

    #  remove acronyms like ICS (Industrial Control Systems)
    text = re.sub(r"\b[A-Z]{2,}\s*\([A-Z]{2,}\)", " ", text)

    # fix broken words
    text = re.sub(r"(\b[\w]{2,})-\s+([\w]{2,}\b)", r"\1\2", text)
    
    # remove legal refs like Stat. 3073
    text = re.sub(r"\bstat\.\s*\d+", " ", text, flags=re.IGNORECASE)

    # remove https fragments
    text = re.sub(r"https?:\S*", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

# -----------------------------
# CHUNKING
# -----------------------------
def split_into_chunks(text, max_sentences=3):
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks = []

    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i+max_sentences]).strip()

        if len(chunk) > 80:
            chunks.append(chunk)

    return chunks

# -----------------------------
# FILTERS
# -----------------------------
def is_noise_title(title):
    t = title.lower()
    return any(x in t for x in [
        "introduction",
        "overview",
        "document",
        "structure",
        "appendix",
        "reference",
        "figure",
        "fig",
        "acronym",
        "glossary",
        "table",
        'stat. 3073. https:'
    ])

def is_bad_heading(title):
    t = title.strip()

    if re.fullmatch(r"[A-Z]{1,3}-\d+", t):
        return True

    if re.fullmatch(r"[A-Z]\.\d+(\.\d+)*\.?", t):
        return True

    if "no ot discussion" in t.lower():
        return True

    if "see https:" in t.lower():
        return True

    if "..." in t:
        return True

    if len(t) < 8:
        return True

    return False

# -----------------------------
# NORMATIVE DETECTION
# -----------------------------
def detect_normative(text):
    text = text.lower()

    if any(x in text for x in ["shall", "must", "required"]):
        return True, "shall"

    if any(x in text for x in ["should", "recommended"]):
        return True, "should"

    if "may" in text:
        return True, "may"

    return False, None

# -----------------------------
# LIFECYCLE CLASSIFICATION
# -----------------------------
LIFECYCLE_DESCRIPTIONS = {
    "Identify": "risk asset inventory governance assessment",
    "Protect": "access control protection encryption safeguards",
    "Detect": "monitor detection logging anomaly detection",
    "Respond": "incident response mitigation communication",
    "Recover": "recovery restore backup resilience"
}

vectorizer = TfidfVectorizer()
phase_names = list(LIFECYCLE_DESCRIPTIONS.keys())
phase_vectors = vectorizer.fit_transform(LIFECYCLE_DESCRIPTIONS.values())

def extract_lifecycle(text, threshold=0.15):
    text_vector = vectorizer.transform([text])
    similarities = cosine_similarity(text_vector, phase_vectors)[0]

    best_idx = similarities.argmax()

    if similarities[best_idx] < threshold:
        return None

    return phase_names[best_idx]

# -----------------------------
# SUMMARY
# -----------------------------
def create_summary(text, max_sentences=2):
    try:
        sentences = nltk.sent_tokenize(text)
    except:
        nltk.download('punkt', quiet=True)
        sentences = nltk.sent_tokenize(text)

    return " ".join(sentences[:max_sentences])

# -----------------------------
# KEYWORDS
# -----------------------------
def extract_keywords(text, max_keywords=5):
    words = re.findall(r'\b[a-zA-Z]{6,}\b', text.lower())

    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1

    sorted_words = sorted(freq.items(), key=lambda x: x[1], reverse=True)

    return [w[0] for w in sorted_words[:max_keywords]]

# -----------------------------
# MAIN EXTRACTION
# -----------------------------
chunks_data = []
seen_chunks = set()

for i, match in enumerate(matches):

    section_number = match.group(1).rstrip(".")
    title = match.group(2).strip()

    # 🔥 FILTER HEADINGS
    if is_noise_title(title) or is_bad_heading(title):
        continue

    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)

    raw_content = full_text[start:end]
    content = clean_content(raw_content)

    if not content:
        continue

    # 🔥 skip noisy content blocks
    if any(x in content.lower() for x in [
        "see table",
        "see figure",
        "see section",
        "acronym",
        "abbreviation"
    ]):
        continue

    # 🔥 skip figure/caption-heavy blocks
    if re.search(r"\bfig(?:ure)?\.?\s*\d+", content.lower()):
        continue

    normative_flag, norm_type = detect_normative(content)

    # 🔥 keep meaningful content
    if not normative_flag and len(content) < 120:
        continue

    summary = create_summary(content)
    keywords = extract_keywords(content)

    chunks = split_into_chunks(content)

    for idx, chunk in enumerate(chunks):

        if chunk in seen_chunks:
            continue
        seen_chunks.add(chunk)

        lifecycle = extract_lifecycle(chunk)

        chunk_obj = {
            "source_standard": "NIST",
            "source_id": f"NIST-{section_number}",
            "section_number": section_number,
            "title": title,
            "parent": section_number.rsplit(".", 1)[0] if "." in section_number else None,
            "chunk_id": f"{section_number}_{idx}",
            "content": chunk,
            "summary": summary,
            "keywords": keywords,
            "lifecycle_phase": lifecycle,
            "normative": normative_flag,
            "normative_type": norm_type
        }

        chunks_data.append(chunk_obj)

# -----------------------------
# SAVE OUTPUT
# -----------------------------
with open(r"D:\exercises\Thesis\nist_sp_800.json", "w", encoding="utf-8") as f:
    json.dump(chunks_data, f, ensure_ascii=False, indent=2)

print(f"✅ Extracted {len(chunks_data)} clean NIST chunks")

✅ Extracted 899 clean NIST chunks


In [47]:
json_path = r"D:\exercises\Thesis\nist_sp_800.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data)
# Rename known standard fields (supports both lower/upper key variants).
rename_map = {
    "standard.name": "Standard Name",
    "standard.version": "Standard Version",
    "standard.year": "Standard Year",
    "standard.type": "Standard Type",
    "standard.Name": "Standard Name",
    "standard.Version": "Standard Version",
    "standard.Year": "Standard Year",
    "standard.Type": "Standard Type",
}

df = df.rename(columns=rename_map)

# Convert list-like keywords to comma-separated text for relational storage.
if "Keywords" in df.columns:
    df["Keywords"] = df["Keywords"].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else x
    )

print(df.head())
print("Columns:", list(df.columns))

  source_standard source_id section_number  \
0            NIST    NIST-2              2   
1            NIST    NIST-2              2   
2            NIST    NIST-2              2   
3            NIST    NIST-2              2   
4            NIST    NIST-2              2   

                                               title parent chunk_id  \
0  OT cybersecurity programs should always be par...    NaN      2_0   
1  OT cybersecurity programs should always be par...    NaN      2_1   
2  OT cybersecurity programs should always be par...    NaN      2_2   
3  OT cybersecurity programs should always be par...    NaN      2_3   
4  OT cybersecurity programs should always be par...    NaN      2_4   

                                             content  \
0  at both industrial sites and enterprise cybers...   
1  Possible incidents that an OT system may face ...   
2  Restrict physical access to the OT network and...   
3  Protect individual OT components from exploita...   
4  Restric

## Transfer to Excel

In [48]:
df.to_excel('NIST_Standard.xlsx', index= False)

In [49]:
# PostgreSQL transfer
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Update these credentials for your environment.
db_user = "postgres"
db_password = "Ruban@1997#"
db_host = "localhost"
db_port = "5432"
db_name = "postgres"
table_name = "nist_standard"

connection_url = (
    f"postgresql+psycopg2://{quote_plus(db_user)}:{quote_plus(db_password)}"
    f"@{db_host}:{db_port}/{db_name}"
)

engine = create_engine(connection_url)

# Writes DataFrame into PostgreSQL; replace table if it already exists.
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Loaded {len(df)} rows into {db_name}.{table_name}")

Loaded 899 rows into postgres.nist_standard


## Transfer to Supabase

In [50]:
import requests
import json 


table_name  = 'nist_standard'

SUPABASE_URL = "https://kaueqrfosfzpdhtnwdko.supabase.co"
SUPABASE_KEY = "sb_publishable_xb-nsEG892PB_8CgyulR0w_T1MVv2Z9"
# API endpoint
url = f"{SUPABASE_URL}/rest/v1/{table_name}"

headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json"
}

delete_url = url + "?chunk_id=not.is.null"  # works if chunk_id exists

delete_response = requests.delete(delete_url, headers=headers)

print("DELETE STATUS:", delete_response.status_code)

# Load JSON
with open(r"D:\exercises\Thesis\nist_sp_800.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Transform data
rows = []
for i, item in enumerate(data):
    standard = item.get("standard", {})

    rows.append({
      "chunk_id": item.get("chunk_id"),

    "standard_name": item.get("source_standard"),  
    "version": "1.0",                              
    "year": 2023,                                 
    "type": "technical",                          

    "source_id": item.get("source_id"),
    "section_number": item.get("section_number"),
    "title": item.get("title"),
    "parent": item.get("parent"),

    "lifecycle_phase": item.get("lifecycle_phase"),
    "normative": item.get("normative"),
    "normative_type": item.get("normative_type"),

    "keywords": item.get("keywords") or [],
    "summary": item.get("summary"),
    "content": item.get("content"),
    })

# Insert into Supabase
response = requests.post(url, headers=headers, json=rows)

print(response.status_code)
print(response.text)

DELETE STATUS: 204
201

